# Thai AI Paper Feed — Phase B: Eval GGUF (Q4_K_M) บน GPU

ปิดช่องว่างที่ค้างจาก Stage 4: **quantize เป็น Q4_K_M แล้วคุณภาพยังดีอยู่ไหม?**

notebook นี้ทำ **อย่างเดียว**: ให้ GGUF (ที่ push ขึ้น HF แล้ว) อ่าน paper 60 ใบใน `test.jsonl` แล้วเขียนสรุป → เก็บเป็น `gen_gguf_q4.jsonl` ลง Drive

การให้คะแนน (LLM-judge + auto-metrics) ทำในเครื่องเหมือนเดิม — ไฟล์นี้แค่ generate

> เทียบกับ `gen_ft_case_a.jsonl` (float LoRA ตัวเดียวกันก่อน quantize) = เห็นเลยว่า Q4 ทำคุณภาพตกกี่แต้ม

**Runtime → เลือก GPU (T4 พอ)** ก่อนรัน

## 0. ติดตั้ง llama-cpp-python (CUDA build)

In [ ]:
%%capture
# build llama-cpp-python พร้อม CUDA — ใช้ GPU offload ทุก layer (เร็วกว่า CPU มาก + ไม่ OOM แบบ Stage 4)
import os
os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
!pip install -q --upgrade llama-cpp-python
!pip install -q huggingface_hub

## 1. โหลด GGUF จาก HF + test set จาก Drive

GGUF ดึงจาก repo ที่ push ไว้ตอน Stage 4 — เท่ากับ eval "ของจริงที่จะเอาไป serve" ไม่ใช่ไฟล์ local ที่อาจต่างจากบน Hub

In [ ]:
from huggingface_hub import hf_hub_download
from google.colab import drive
import os

REPO_ID = "Tana-Pun/thai-paper-summarizer-7b"
GGUF_FILE = "typhoon2-qwen2.5-7b-instruct.Q4_K_M.gguf"

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/thai-paper-feed-phase-b'
EVAL_DIR = f'{DRIVE_ROOT}/eval'
os.makedirs(EVAL_DIR, exist_ok=True)

GGUF_PATH = hf_hub_download(repo_id=REPO_ID, filename=GGUF_FILE)
print('GGUF ->', GGUF_PATH)
print('EVAL_DIR ->', EVAL_DIR)

In [ ]:
import json

def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

test_rows = load_jsonl(f'{DRIVE_ROOT}/data/test.jsonl')
print(f'test: {len(test_rows)} rows')
print('ตัวอย่าง id:', [r['id'] for r in test_rows[:3]])

## 2. โหลดโมเดล + ฟังก์ชัน generate

ตั้งให้ **ตรงกับ eval float เดิม** ให้มากที่สุด เพื่อเทียบ apple-to-apple:
- **greedy** (`temperature=0`) — ผลซ้ำได้ ไม่งั้นเทียบข้ามโมเดลไม่ได้
- `max_tokens=1024` เท่าเดิม
- ใช้ chat template ที่ฝังใน GGUF (typhoon2-qwen2.5 = ChatML) ผ่าน `create_chat_completion`
- เก็บ `raw` ดิบทั้งหมด (JSON พังก็เก็บ — เป็นตัวชี้วัด)
- **resume ได้** — หลุดแล้วรันซ้ำ ข้ามใบที่ทำแล้ว
- `n_gpu_layers=-1` offload ทุก layer ขึ้น GPU, `n_ctx=4096` พอสำหรับ prompt ~1.1k + gen 1024

In [ ]:
from llama_cpp import Llama
import time

llm = Llama(
    model_path=GGUF_PATH,
    n_gpu_layers=-1,      # offload ทุก layer ขึ้น GPU
    n_ctx=4096,
    verbose=False,
)
print('โหลดโมเดลเสร็จ')

In [ ]:
def generate_all(out_name="gen_gguf_q4", max_tokens=1024):
    out_path = f'{EVAL_DIR}/{out_name}.jsonl'

    # resume: ข้ามใบที่เคยทำแล้ว
    done = set()
    if os.path.exists(out_path):
        done = {r['id'] for r in load_jsonl(out_path)}
        print(f'มีผลเดิม {len(done)} ใบ -> ทำต่อ')

    todo = [r for r in test_rows if r['id'] not in done]
    if not todo:
        print('ครบแล้ว ไม่มีอะไรต้องทำ')
        return out_path

    print(f'=== gguf_q4 === เหลือ {len(todo)} ใบ')
    with open(out_path, 'a', encoding='utf-8') as f:
        for i, row in enumerate(todo, 1):
            messages = [
                {"role": "system", "content": row["system"]},
                {"role": "user", "content": row["user"]},
            ]
            t0 = time.perf_counter()
            resp = llm.create_chat_completion(
                messages=messages,
                max_tokens=max_tokens,
                temperature=0.0,   # greedy
            )
            elapsed = time.perf_counter() - t0
            text = resp["choices"][0]["message"]["content"]
            f.write(json.dumps({
                "id": row["id"],
                "system_name": "gguf_q4",
                "raw": text,
                "sec": round(elapsed, 2),
            }, ensure_ascii=False) + "\n")
            f.flush()
            if i % 10 == 0 or i == len(todo):
                print(f'  {i}/{len(todo)}  ({elapsed:.1f} วิ/ใบ)')

    print(f'เสร็จ -> {out_path}')
    return out_path

### ลอง 2 ใบก่อน (กันพัง/กัน template เพี้ยน ก่อนวน 60)

In [ ]:
# sanity 2 ใบ — เช็คว่า output เป็น JSON การ์ดไทยจริง ไม่ใช่ template เพี้ยน/ภาษาอังกฤษ
for row in test_rows[:2]:
    resp = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": row["system"]},
            {"role": "user", "content": row["user"]},
        ],
        max_tokens=1024, temperature=0.0,
    )
    print('id', row['id'])
    print(resp["choices"][0]["message"]["content"][:400])
    print('-' * 60)

### รันเต็ม 60 ใบ

In [ ]:
out_path = generate_all()

## 3. เช็คเร็วๆ ก่อนปิด GPU

JSON valid % + field ครบ + เวลา/ใบ ของ **gguf_q4 เทียบ ft_case_a (float)** ที่ทำไว้แล้ว
(ยังไม่ใช่คะแนนจริง — judge ทำในเครื่อง)

In [ ]:
import re
REQ = {"title_th", "summary_th", "wow_point", "tags"}

def try_parse(raw):
    try:
        return json.loads(raw)
    except Exception:
        pass
    m = re.search(r"\{.*\}", raw, re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return None

def stat(fn):
    rows = load_jsonl(f'{EVAL_DIR}/{fn}')
    ok = sum(try_parse(r["raw"]) is not None for r in rows)
    fok = sum(isinstance(try_parse(r["raw"]), dict) and REQ.issubset(try_parse(r["raw"])) for r in rows)
    avg = sum(r.get("sec", 0) for r in rows) / len(rows)
    return len(rows), ok, fok, avg

print(f"{'ระบบ':<12}{'ใบ':>4}{'JSON valid':>14}{'field ครบ':>12}{'วิ/ใบ':>9}")
print('-' * 51)
for fn in ['gen_gguf_q4.jsonl', 'gen_ft_case_a.jsonl']:
    if os.path.exists(f'{EVAL_DIR}/{fn}'):
        n, ok, fok, avg = stat(fn)
        name = fn[len('gen_'):-len('.jsonl')]
        print(f"{name:<12}{n:>4}{ok:>7}/{n:<2}({100*ok//n:>3}%){fok:>7}/{n:<2}{avg:>9.1f}")

## เสร็จแล้วทำอะไรต่อ

1. โหลด `gen_gguf_q4.jsonl` จาก `MyDrive/thai-paper-feed-phase-b/eval/` ลงเครื่อง → วางที่ `phase-b/data/eval/`
2. ในเครื่อง (ไม่ต้องแตะ GPU อีก):
   - `python phase-b/scripts/eval_auto_metrics.py` — structure/tags/ศัพท์
   - `python phase-b/scripts/eval_llm_judge.py` — judge หยิบ gguf_q4 มาให้คะแนนอัตโนมัติ (resume ของเดิม)
   - ทำตารางเทียบ **gguf_q4 vs ft_case_a** → รู้ว่า Q4 ตกกี่แต้ม ผ่านเกณฑ์ไหม → ไฟเขียวไป Stage 5